# 第九课：同步、异步与并发基础

本课目标：

- 理解同步等待为什么浪费时间；
- 看懂`async def`、协程和`await`；
- 使用`asyncio.sleep()`模拟网络等待；
- 使用`asyncio.gather()`并发处理多个请求；
- 理解异步适合I/O等待，不会自动加速CPU计算；
- 使用pytest测试异步函数。

以后同时请求多个模型、下载多个网页或访问数据库时会用到这些概念。

## 1. 什么是同步执行

同步执行表示一件事完成后，才开始下一件事：

```text
请求A：等待1秒
请求B：等待1秒
请求C：等待1秒
总时间约3秒
```

下面使用`time.sleep()`模拟阻塞等待。

In [ ]:
import time


def sync_request(name: str, delay: float) -> str:
    print(name, "开始")
    time.sleep(delay)
    print(name, "结束")
    return name


start = time.perf_counter()
results = [
    sync_request("A", 0.3),
    sync_request("B", 0.3),
    sync_request("C", 0.3),
]
seconds = time.perf_counter() - start

print(results)
print(f"用时：{seconds:.2f}秒")

## 2. `async def`和协程

使用`async def`定义异步函数：

```python
async def async_request():
    ...
```

调用普通函数会立即执行函数体；调用异步函数首先得到一个**协程对象**，它描述一项可以被异步调度的工作。

In [ ]:
import asyncio


async def say_hello() -> str:
    return "hello"


coroutine = say_hello()
print(coroutine)
print(type(coroutine))

# 关闭这个尚未运行的协程，避免Notebook产生警告。
coroutine.close()

## 3. `await`是什么意思

`await`表示等待一个异步操作完成。在等待网络、磁盘或定时器时，事件循环可以先运行其他任务。

`await`通常只能写在`async def`内部。Jupyter为了方便，也允许在代码单元格顶层直接使用`await`。

In [ ]:
async def async_request(name: str, delay: float) -> str:
    print(name, "开始")
    await asyncio.sleep(delay)
    print(name, "结束")
    return name


result = await async_request("A", 0.3)
print(result)

### `time.sleep()`和`asyncio.sleep()`的区别

```python
time.sleep(1)
```

会阻塞当前线程，事件循环不能利用这段等待时间。

```python
await asyncio.sleep(1)
```

会暂停当前协程并把执行机会交还事件循环，让其他任务先运行。

异步函数中不要随意使用`time.sleep()`。

## 4. 写了`async`不代表自动并发

下面虽然使用异步函数，但每次都立刻`await`，所以仍然按顺序执行：

In [ ]:
start = time.perf_counter()

sequential_results = []
for name in ["A", "B", "C"]:
    result = await async_request(name, 0.3)
    sequential_results.append(result)

seconds = time.perf_counter() - start
print(sequential_results)
print(f"顺序异步用时：{seconds:.2f}秒")

## 5. 使用`asyncio.gather()`并发等待

先创建多项协程工作，再把它们一起交给`gather()`：

In [ ]:
start = time.perf_counter()

coroutines = [
    async_request("A", 0.3),
    async_request("B", 0.3),
    async_request("C", 0.3),
]
concurrent_results = await asyncio.gather(*coroutines)

seconds = time.perf_counter() - start
print(concurrent_results)
print(f"并发用时：{seconds:.2f}秒")

`*coroutines`表示把列表中的多个协程作为独立参数传给`gather()`：

```python
asyncio.gather(*coroutines)
```

近似于：

```python
asyncio.gather(coroutine_a, coroutine_b, coroutine_c)
```

`gather()`返回的结果顺序通常与传入协程的顺序一致，不取决于谁先完成。

## 6. 异步适合什么任务

适合大量等待外部响应的I/O任务：

- 调用大模型API；
- 下载网页；
- 访问数据库；
- 读写网络连接。

异步不会自动加速纯CPU计算，例如大型矩阵运算、图片编码或模型推理。CPU密集任务通常需要多进程、GPU或专门的任务系统。

## 7. 普通Python文件怎样启动异步程序

Notebook可以直接写`await`，普通`.py`文件通常使用：

```python
async def main():
    result = await async_request("A", 0.1)
    print(result)


if __name__ == "__main__":
    asyncio.run(main())
```

`asyncio.run()`创建事件循环、运行协程，并在完成后关闭事件循环。

## 8. 使用pytest测试异步函数

本课不额外安装异步测试插件，而是在普通测试中使用`asyncio.run()`：

```python
def test_async_function():
    result = asyncio.run(mock_llm_call("你好", delay=0))
    assert result == "回答：你好"
```

这样仍然是普通pytest测试函数。

## 9. 动手作业

文件：

- `async_service.py`：完成三个异步函数；
- `test_async_service.py`：已有测试，并需要补两个测试。

步骤：

1. 运行`python -m pytest -v`观察初始失败；
2. 完成`mock_llm_call()`；
3. 完成`run_sequential()`；
4. 完成`run_concurrent()`；
5. 补充测试文件末尾的两个测试；
6. 反复运行直到全部通过。

In [ ]:
import pytest

exit_code = pytest.main(["-v", "test_async_service.py"])
print("pytest退出码：", exit_code)
assert exit_code == pytest.ExitCode.OK

## 10. 理解题

1. 调用`async def`函数时，为什么不会像普通函数一样立即得到最终返回值？
2. `await`的作用是什么？
3. 为什么在异步函数中通常不应该使用`time.sleep()`？
4. 仅仅把函数写成`async def`，多个调用就会自动并发吗？为什么？
5. `asyncio.gather()`解决了什么问题？返回结果顺序如何确定？
6. 为什么异步适合调用大模型API，但不会自动加速CPU密集型模型推理？